# Reconstruct direct asymptotic and patient-bootstrap Oλ
Reuse the patient assignments from the original repeated reconstruction. Fit ridge, elastic net, PLS and shallow gradient boosting using TCAT GEPs, atlas hard-label cell types and the two representations together. Both targets, tuning MAE and held-out predictions are on the observed Oλ scale. The bootstrap target is the patient mean of the 25 replicate reciprocals. Each patient appears once per repeat. No survival outputs are required.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import tempfile
import warnings
import numpy as np
import pandas as pd
import sklearn
from scipy.stats import rankdata
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import StratifiedKFold
from IPython.display import display

drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/Hartert/CAR-T-spectral-organization/DATA')
TCAT = DATA/'patient_TCAT_particles/all_patient_TCAT_estimates_harmonized.csv'
ATLAS = DATA/'Functional_CART_atlas_patient_particles_hard_labels_all8/patient_atlas_estimates.csv'
DIRECT = DATA/'olambda_reciprocal_direct_no_asymptote_floor_v1/patient_olambda_asymptotes.csv'
OLD = DATA/'olambda_reconstruction_from_TCAT_and_atlas/repeated_cross_validated_predictions.csv'
BOOT_DIR = DATA/'olambda_sensitivity/patient_bayesian_bootstrap'
BOOT = BOOT_DIR/'olambda_sensitivity_replicates.csv'
OUT = DATA/'direct_Olambda_reconstruction_from_TCAT_and_atlas'
SEED = 0
REVISION = '2fdfe049789d1967cdd72ac682950d3743c13b69'
repo = Path(tempfile.mkdtemp())/'repo'
subprocess.run(['git','clone','--quiet','--single-branch','--branch','direct-olambda',
    'https://github.com/adamnik/CAR-T-spectral-organization.git',repo],check=True)
subprocess.run(['git','-C',repo,'checkout','--quiet',REVISION],check=True)
sys.path.insert(0,str(repo/'cell_state_relationships'))
from reconstruct_olambda_from_cell_states import (
    MODELS, average_patient_predictions, centered_geometry, make_estimator, parameter_grid, predict,
)

In [ ]:
keys = ['Dataset','sample_id']
def features(path, prefix, count):
    table = pd.read_csv(path).set_index(keys, verify_integrity=True)
    cols = [c for c in table if c.startswith(prefix) and c.endswith('_CLR')]
    assert len(cols) == count, (path, cols)
    return table[cols]
tc = features(TCAT,'TCAT_GEP_',52)
at = features(ATLAS,'ATLAS_proportion_',11)
direct = pd.read_csv(DIRECT).set_index(keys,verify_integrity=True)
assert direct.olambda_scale.eq('reciprocal').all()
assert direct.model.eq('bounded_direct_no_asymptote_floor').all()
targets = ['direct_asymptotic_Olambda','patient_bootstrap_Olambda']
columns = keys + ['experiment','replicate','O_lambda','n_cells_target']
files = [BOOT] if BOOT.exists() else sorted((BOOT_DIR/'replicate_parts').glob('*.csv'))
if not files:
    raise FileNotFoundError(f'Set BOOT to the historical patient bootstrap replicate CSV: {BOOT}')
raw = pd.concat([pd.read_csv(f,usecols=columns) for f in files], ignore_index=True)
assert raw.experiment.eq('patient-bootstrap').all() and raw.n_cells_target.eq(10000).all()
replicates = raw.pivot(index=keys,columns='replicate',values='O_lambda')
assert list(replicates.columns)==list(range(25))
assert np.isfinite(replicates).all().all() and (replicates>1).all().all()
data = direct[['O_lambda_inf_central']].rename(columns={'O_lambda_inf_central':targets[0]})
data = data.join((1/replicates).mean(axis=1).rename(targets[1])).join(tc).join(at)
assert np.isfinite(data).all().all()
assert ((data[targets]>0)&(data[targets]<1)).all().all()
data = data.sort_index().reset_index()
assert len(data)==163 and data.Dataset.nunique()==8
groups = {'TCAT GEP':list(tc), 'Atlas cell type':list(at), 'TCAT GEP + Atlas':[*tc,*at]}

# The prior predictions are only used for identifiers and fold membership.
folds = pd.read_csv(OLD,usecols=keys+['repeat','fold']).drop_duplicates()
assert set(folds.repeat)=={0,1,2}
splits = []
for repeat, part in folds.groupby('repeat'):
    membership = data[keys].merge(part,on=keys,how='outer',validate='one_to_one',indicator=True)
    assert membership._merge.eq('both').all() and set(membership.fold)==set(range(4))
    by_patient = part.set_index(keys).fold.reindex(pd.MultiIndex.from_frame(data[keys])).to_numpy()
    for fold in range(4):
        train,test = np.flatnonzero(by_patient!=fold),np.flatnonzero(by_patient==fold)
        assert set(data.iloc[test].Dataset)<=set(data.iloc[train].Dataset)
        splits.append((int(repeat),fold,train,test))
config = dict(revision=REVISION, sklearn=sklearn.__version__, seed=SEED,
              target_scale='direct', metric_bootstraps=1000, version=1)
digest = hashlib.sha256(json.dumps(config,sort_keys=True).encode())
for table in [data,folds.sort_values(['repeat',*keys])]:
    digest.update(table.to_csv(index=False).encode())
RUN = OUT/digest.hexdigest()[:12]
RUN.mkdir(parents=True,exist_ok=True)
(RUN/'config.json').write_text(json.dumps(config,indent=2))
folds.to_csv(RUN/'patient_fold_assignments.csv',index=False)
data[keys+targets].to_csv(RUN/'reconstruction_targets.csv',index=False)
print('Reusing 3 repeats × 4 folds for',len(data),'patients. Output:',RUN,flush=True)
display(data.groupby('Dataset').size().rename('patients'))

In [ ]:
# Within every training split, center by the training patients from each dataset.
# The estimator pipeline also learns feature scaling from that training split.
def fit_fold(train, test, features, y, repeat, fold):
    splitter = StratifiedKFold(n_splits=3,shuffle=True,random_state=1000*repeat+10*fold+SEED)
    inner = []
    for a,b in splitter.split(train,train.Dataset):
        tr,va = train.iloc[a],train.iloc[b]
        assert set(va.Dataset)<=set(tr.Dataset)
        inner.append((*centered_geometry(tr,va,features,y),y[va.index]))
    x,xt,yt,offset = centered_geometry(train,test,features,y)
    predictions = {'Dataset mean':offset}
    tuning = []
    for model in MODELS:
        best_error = np.inf
        for params in parameter_grid(model,len(features),len(train),False):
            errors = []
            for xi,xv,yi,off,actual in inner:
                estimator = make_estimator(model,params,SEED).fit(xi,yi)
                errors.append(mean_absolute_error(actual,predict(estimator,xv)+off))
            error = np.mean(errors)
            if error<best_error:
                best_error,best = error,params
        predictions[model] = predict(make_estimator(model,best,SEED).fit(x,yt),xt)+offset
        tuning.append(dict(model=model,inner_MAE_Olambda=best_error,best_parameters=json.dumps(best)))
    return predictions,pd.DataFrame(tuning)

prediction_parts,tuning_parts = [],[]
for rep,features_used in groups.items():
    for target in targets:
        y = data[target].to_numpy()
        for repeat,fold,a,b in splits:
            dest = RUN/'folds'/rep/target/f'r{repeat}_f{fold}'
            print(rep,target,repeat+1,fold+1,'(saved)' if dest.exists() else '(fitting)',flush=True)
            if not dest.exists():
                pred,tune = fit_fold(data.iloc[a],data.iloc[b],features_used,y,repeat,fold)
                identifiers = dict(representation=rep,target=target,repeat=repeat,fold=fold)
                frame = pd.concat([data.iloc[b][keys].assign(row=b,model=model,observed=y[b],
                    predicted=values,**identifiers) for model,values in pred.items()])
                dest.parent.mkdir(parents=True,exist_ok=True)
                stage = Path(tempfile.mkdtemp(dir=dest.parent))
                frame.to_csv(stage/'predictions.csv',index=False)
                tune.assign(**identifiers).to_csv(stage/'tuning.csv',index=False)
                stage.rename(dest)
            prediction_parts.append(pd.read_csv(dest/'predictions.csv'))
            tuning_parts.append(pd.read_csv(dest/'tuning.csv'))
repeated = pd.concat(prediction_parts,ignore_index=True)
assert len(repeated)==len(data)*3*len(groups)*len(targets)*(len(MODELS)+1)
patient = average_patient_predictions(repeated)
repeated.to_csv(RUN/'repeated_cross_validated_predictions.csv',index=False)
patient.to_csv(RUN/'patient_cross_validated_predictions.csv',index=False)
pd.concat(tuning_parts).to_csv(RUN/'nested_tuning_results.csv',index=False)

## Held-out performance
Average the three held-out predictions per patient. Report ordinary Oλ-scale R² and dataset-adjusted Spearman correlation. For the latter, rank observed and predicted values across patients and remove the dataset mean of each rank before correlating them. Resample patients within datasets for percentile intervals, using the same resamples across models. Intervals condition on the fitted cross-validation predictions.

In [ ]:
codes = pd.factorize(data.Dataset)[0]
indices = [np.flatnonzero(codes==i) for i in np.unique(codes)]
rng = np.random.default_rng(SEED)
samples = [np.concatenate([rng.choice(idx,len(idx)) for idx in indices]) for _ in range(1000)]
def metrics(y,p,g):
    ranks = rankdata(np.column_stack([y,p]),axis=0)
    for i in np.unique(g):
        mask = g==i
        ranks[mask] -= ranks[mask].mean(axis=0)
    denom = np.linalg.norm(ranks[:,0])*np.linalg.norm(ranks[:,1])
    rho = ranks[:,0]@ranks[:,1]/denom if denom else np.nan
    return r2_score(y,p),rho
rows = []
for (target,rep,model), frame in patient.groupby(['target','representation','model']):
    frame = frame.sort_values('row')
    np.testing.assert_array_equal(frame.row,np.arange(len(data)))
    y,p = frame.observed.to_numpy(),frame.predicted.to_numpy()
    point = metrics(y,p,codes)
    draws = [metrics(y[ix],p[ix],codes[ix]) for ix in samples]
    lo,hi = np.nanquantile(draws,[.025,.975],axis=0)
    rows.append(dict(target=target,representation=rep,model=model,patients=len(y),
        R2=point[0],R2_q025=lo[0],R2_q975=hi[0],
        dataset_adjusted_Spearman_r=point[1],r_q025=lo[1],r_q975=hi[1]))
summary = pd.DataFrame(rows).sort_values(['target','representation','R2'],ascending=[True,True,False])
summary.to_csv(RUN/'reconstruction_summary.csv',index=False)
display(summary.round(3))
print(RUN/'reconstruction_summary.csv')